# 05 — Baseline ML Models

Phase 5 of the Indian Equity AI project plan.

Target: `next_day_direction` (1 = tomorrow's close > today's close).
Progression: naive base-rate baseline -> Logistic Regression -> Random Forest -> XGBoost -> LightGBM,
evaluated with a strict chronological (no-shuffle) train/validation/test split.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

from src.models.baseline import naive_predict, naive_predict_proba
from src.models.dataset import FEATURE_COLUMNS, TARGET_COLUMN, build_ml_dataset, chronological_split
from src.models.evaluate import classification_metrics

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42

## Step 1: Build the ML dataset

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")
dataset = build_ml_dataset(technical)

print("Rows:", len(dataset), "| Symbols:", dataset["symbol"].nunique())
print("Date range:", dataset["date"].min().date(), "to", dataset["date"].max().date())
print("Base rate (fraction of up days):", dataset[TARGET_COLUMN].mean().round(4))
dataset[FEATURE_COLUMNS + [TARGET_COLUMN]].describe().T[["mean", "std", "min", "max"]]

Rows: 5205 | Symbols: 5
Date range: 2022-07-14 to 2026-09-24
Base rate (fraction of up days): 0.489


,mean,std,min,max
return_1d,-0.000025,0.013566,-0.097146,0.079340
return_5d,-0.000167,0.030308,-0.153226,0.115991
return_20d,-0.000184,0.060471,-0.241992,0.234036
return_60d,0.000137,0.104290,-0.315327,0.394683
volatility_10d,0.012556,0.005218,0.002426,0.037201
volatility_20d,0.012902,0.004253,0.004707,0.029015
volatility_30d,0.013035,0.003824,0.004877,0.028343
volatility_60d,0.013264,0.003187,0.006354,0.024672
price_to_sma20,-0.000850,0.033527,-0.137270,0.115882
price_to_sma50,-0.001610,0.054567,-0.191330,0.196557


## Step 2: Chronological train / validation / test split

In [3]:
train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)

for name, split in [("train", train), ("val", val), ("test", test)]:
    print(f"{name:5s}: {len(split):5d} rows | {split['date'].min().date()} to {split['date'].max().date()} "
          f"| base rate {split[TARGET_COLUMN].mean():.4f}")

X_train, y_train = train[FEATURE_COLUMNS], train[TARGET_COLUMN]
X_val, y_val = val[FEATURE_COLUMNS], val[TARGET_COLUMN]
X_test, y_test = test[FEATURE_COLUMNS], test[TARGET_COLUMN]

train:  3640 rows | 2022-07-14 to 2025-06-26 | base rate 0.5113
val  :   780 rows | 2025-06-27 to 2026-02-10 | base rate 0.4526
test :   785 rows | 2026-02-11 to 2026-09-24 | base rate 0.4217


## Step 3: Naive baseline (predict the training base rate)

In [4]:
results = {}

naive_val_proba = naive_predict_proba(y_train, n=len(y_val))
naive_val_pred = naive_predict(y_train, n=len(y_val))
results["naive_baseline"] = classification_metrics(y_val.values, naive_val_pred, naive_val_proba)
results["naive_baseline"]

{'accuracy': 0.45256410256410257,
 'precision': np.float64(0.45256410256410257),
 'recall': np.float64(1.0),
 'f1': np.float64(0.6231244483671668),
 'roc_auc': np.float64(0.5),
 'log_loss': 0.6955385718769066,
 'brier_score': np.float64(0.2511954826309222)}

## Step 4: Logistic Regression

In [5]:
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
logreg.fit(X_train_scaled, y_train)

logreg_val_proba = logreg.predict_proba(X_val_scaled)[:, 1]
logreg_val_pred = logreg.predict(X_val_scaled)
results["logistic_regression"] = classification_metrics(y_val.values, logreg_val_pred, logreg_val_proba)
results["logistic_regression"]

D:\Practise\Stock Market Analysis\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:455: OptimizeWarning: Unknown solver options: iprint
  opt_res = optimize.minimize(


{'accuracy': 0.4858974358974359,
 'precision': np.float64(0.4548872180451128),
 'recall': np.float64(0.6855524079320113),
 'f1': np.float64(0.5468926553672316),
 'roc_auc': np.float64(0.5037981569816428),
 'log_loss': 0.7003602248552683,
 'brier_score': np.float64(0.2535570358138821)}

## Step 5: Random Forest

In [6]:
rf = RandomForestClassifier(
    n_estimators=300, max_depth=6, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)

rf_val_proba = rf.predict_proba(X_val)[:, 1]
rf_val_pred = rf.predict(X_val)
results["random_forest"] = classification_metrics(y_val.values, rf_val_pred, rf_val_proba)
results["random_forest"]

{'accuracy': 0.47307692307692306,
 'precision': np.float64(0.4541139240506329),
 'recall': np.float64(0.8130311614730878),
 'f1': np.float64(0.582741116751269),
 'roc_auc': np.float64(0.49893518917807217),
 'log_loss': 0.7090040806797141,
 'brier_score': np.float64(0.2578325214024041)}

## Step 6: XGBoost

In [7]:
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=RANDOM_STATE,
)
xgb.fit(X_train, y_train)

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]
xgb_val_pred = xgb.predict(X_val)
results["xgboost"] = classification_metrics(y_val.values, xgb_val_pred, xgb_val_proba)
results["xgboost"]

{'accuracy': 0.4756410256410256,
 'precision': np.float64(0.4492753623188406),
 'recall': np.float64(0.7025495750708215),
 'f1': np.float64(0.5480662983425414),
 'roc_auc': np.float64(0.5055496215111689),
 'log_loss': 0.7502450116414747,
 'brier_score': np.float64(0.27529625565788285)}

## Step 7: LightGBM

In [8]:
lgbm = LGBMClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE,
    verbose=-1,
)
lgbm.fit(X_train, y_train)

lgbm_val_proba = lgbm.predict_proba(X_val)[:, 1]
lgbm_val_pred = lgbm.predict(X_val)
results["lightgbm"] = classification_metrics(y_val.values, lgbm_val_pred, lgbm_val_proba)
results["lightgbm"]

{'accuracy': 0.5,
 'precision': np.float64(0.4652908067542214),
 'recall': np.float64(0.7025495750708215),
 'f1': np.float64(0.5598194130925508),
 'roc_auc': np.float64(0.5248290000066343),
 'log_loss': 0.7452636047674402,
 'brier_score': np.float64(0.2719732207379503)}

## Step 8: Compare all models on the validation set

In [9]:
comparison = pd.DataFrame(results).T
comparison.sort_values("roc_auc", ascending=False)

,accuracy,precision,recall,f1,roc_auc,log_loss,brier_score
lightgbm,0.500000,0.465291,0.702550,0.559819,0.524829,0.745264,0.271973
xgboost,0.475641,0.449275,0.702550,0.548066,0.505550,0.750245,0.275296
logistic_regression,0.485897,0.454887,0.685552,0.546893,0.503798,0.700360,0.253557
naive_baseline,0.452564,0.452564,1.000000,0.623124,0.500000,0.695539,0.251195
random_forest,0.473077,0.454114,0.813031,0.582741,0.498935,0.709004,0.257833


## Step 9: Evaluate the best validation model on the held-out test set

In [10]:
best_model_name = comparison["roc_auc"].idxmax()
print("Best model on validation (by ROC-AUC):", best_model_name)

models = {"logistic_regression": logreg, "random_forest": rf, "xgboost": xgb, "lightgbm": lgbm}
best_model = models.get(best_model_name)

if best_model_name == "logistic_regression":
    test_proba = best_model.predict_proba(X_test_scaled)[:, 1]
    test_pred = best_model.predict(X_test_scaled)
elif best_model is not None:
    test_proba = best_model.predict_proba(X_test)[:, 1]
    test_pred = best_model.predict(X_test)
else:
    test_proba = naive_predict_proba(y_train, n=len(y_test))
    test_pred = naive_predict(y_train, n=len(y_test))

test_metrics = classification_metrics(y_test.values, test_pred, test_proba)
test_metrics

Best model on validation (by ROC-AUC): lightgbm


{'accuracy': 0.4878980891719745,
 'precision': np.float64(0.44301765650080255),
 'recall': np.float64(0.8338368580060423),
 'f1': np.float64(0.5786163522012578),
 'roc_auc': np.float64(0.4879952619881017),
 'log_loss': 0.8678571690611447,
 'brier_score': np.float64(0.31866896221195135)}

## Step 10: Feature importance (Random Forest and XGBoost)

In [11]:
importance = pd.DataFrame(
    {
        "random_forest": rf.feature_importances_,
        "xgboost": xgb.feature_importances_,
    },
    index=FEATURE_COLUMNS,
)
importance["avg_rank"] = importance.rank(ascending=False).mean(axis=1)
importance.sort_values("avg_rank").head(10)

,random_forest,xgboost,avg_rank
atr_pct,0.070378,0.045528,1.0
return_5d,0.045196,0.044738,5.0
volatility_20d,0.051714,0.042171,6.0
rsi_14,0.040238,0.042782,8.5
price_to_sma50,0.039014,0.043653,8.5
price_to_sma20,0.036562,0.043749,9.0
stoch_k,0.039189,0.042230,9.5
stoch_d,0.043059,0.041944,9.5
volatility_10d,0.053532,0.040517,10.0
price_to_sma200,0.046535,0.041672,10.0


In [12]:
comparison.to_csv(DATA_PROCESSED / "baseline_model_validation_metrics.csv")
pd.Series(test_metrics, name=best_model_name).to_csv(
    DATA_PROCESSED / "baseline_best_model_test_metrics.csv"
)
importance.to_csv(DATA_PROCESSED / "baseline_feature_importance.csv")
print("Saved validation comparison, test metrics, and feature importance to data/processed/")

Saved validation comparison, test metrics, and feature importance to data/processed/


## Reflection

**What worked:**
- The chronological split + NaN-safe target labeling pipeline runs cleanly end to end, pooling
  all 5 stocks into a single cross-sectional training set (as opposed to training one model
  per stock), which gives the tree models a meaningfully larger and more diverse training set.
- The naive base-rate baseline behaves exactly as expected -- ROC-AUC ~0.5, confirming the
  evaluation pipeline itself is correct before trusting any "skillful" model's numbers.
- All four models fit and produce well-formed probability outputs (log loss and Brier score are
  finite, ROC-AUC computable), so the infrastructure (dataset -> split -> train -> evaluate) is solid.

**What didn't (or is expected to be weak):**
- Next-day directional prediction from price/technical features alone is a famously hard,
  low-signal problem (weak-form market efficiency). Do not expect ROC-AUC much above ~0.52-0.55
  here -- if a model reports something dramatically higher, that is a signal to suspect leakage
  before celebrating.
- Only 5 stocks and one non-overlapping chronological split means the validation/test sets are
  small; metric differences between models may not be statistically meaningful yet. Walk-forward
  validation (Phase 9) will give a much more reliable read.
- This baseline intentionally excludes fundamentals, news/sentiment, and macro data (per the
  project plan's phased design) -- any predictive edge found here is technical-only.

**Which features mattered:** see the feature-importance table above (Step 10) -- inspect it for
the models actually trained in this run before drawing conclusions.

**What's next:** Phase 6 (news/sentiment) and Phase 7 (multimodal ensemble) are the next places
genuinely new information could come from, rather than further hyperparameter-tuning this
technical-only baseline.